In [ ]:
import copernicusmarine
import gsw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.interpolate import interp1d
import xarray as xr

In [ ]:
# Download one month of GLORYS reanalysis for North Pacific
glorys_ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "so"],
    minimum_longitude=-180,
    maximum_longitude=-130,
    minimum_latitude=30,
    maximum_latitude=60,
    minimum_depth=0,
    maximum_depth=1000,
    start_datetime="2020-01-01",
    end_datetime="2020-01-31",
)
print(glorys_ds)


In [ ]:
# Calculate density
SA = gsw.SA_from_SP(
    glorys_ds['so'],
    glorys_ds['depth'],
    glorys_ds['longitude'],
    glorys_ds['latitude']
)

CT = gsw.CT_from_pt(
    SA,
    glorys_ds['thetao']
)

sigma_theta = gsw.sigma0(SA, CT)

glorys_ds['sigma_theta'] = (('time', 'depth', 'latitude', 'longitude'), sigma_theta.data)


In [ ]:
# Plot temperature-density profiles for one day

# 5deg grid
lats = np.arange(30, 61, 10)
lons = np.arange(-180, -131, 10)

# Fetch profiles
profiles = (
    glorys_ds
    .sel(time='2020-01-01')
    .sel(latitude=lats, longitude=lons, method='nearest')
    .compute()
)

# Set up colormap normalized to lon range
norm = mcolors.Normalize(vmin=lats.min(), vmax=lats.max())
cmap = cm.coolwarm_r

fig, ax = plt.subplots(figsize=(8, 6))
for lat in lats:
    for lon in lons:
        profile = profiles.sel(latitude=lat, longitude=lon)
        ax.plot(profile['thetao'], 
                profile['sigma_theta'],
                color=cmap(norm(lat)))

ax.set_ylim(28, 24)
ax.set_xlabel('Potential Temperature (°C)')
ax.set_ylabel('σ₀ (kg/m³)')
ax.set_title('T–σ₀ profiles, Jan 1 2020')

# Add colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='Latitude')

plt.tight_layout()
plt.show()


We see a clear bifurcation by latitude. The cluster of profiles on the left come from high latitudes. They have lower temperature for a given density, meaning they're fresher. The right group of profiles are lower latitude and saltier at a given isopycnal.

In [ ]:
# Plot salinity-depth profiles for the same day

# 5deg grid
lats = np.arange(30, 61, 10)
lons = np.arange(-180, -131, 10)

# Fetch profiles
profiles = (
    glorys_ds
    .sel(time='2020-01-01')
    .sel(latitude=lats, longitude=lons, method='nearest')
    .compute()
)

# Set up colormap normalized to lon range
norm = mcolors.Normalize(vmin=lats.min(), vmax=lats.max())
cmap = cm.coolwarm_r

fig, ax = plt.subplots(figsize=(8, 6))
for lat in lats:
    for lon in lons:
        profile = profiles.sel(latitude=lat, longitude=lon)
        ax.plot(profile['so'], 
                profile['depth'],
                color=cmap(norm(lat)))

ax.invert_yaxis()
ax.set_xlabel('Salinity (psu)')
ax.set_ylabel('Depth (m)')
ax.set_title('S–D profiles, Jan 1 2020')

# Add colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='Latitude')

plt.tight_layout()
plt.show()

Here we clearly see the effect of fresh water entering at high latitudes.

What we're going to do next is focus in on the 26.8 isopycnal, plotting the salinity at the isopycnal and its depth in two panels.

In [ ]:
# Interpolation function
def interp_to_isopycnal(sigma, val, target):
    mask = ~np.isnan(sigma) & ~np.isnan(val)
    if np.sum(mask) < 2:
        return np.nan
    s, v = sigma[mask], val[mask]
    if target < s.min() or target > s.max():
        return np.nan
    try:
        return interp1d(s, v)(target)
    except:
        return np.nan
    
day = glorys_ds.sel(time='2020-01-01')

# Get salinity on 26.8 isopycnal
sal_iso = xr.apply_ufunc(
    interp_to_isopycnal,
    day['sigma_theta'],
    day['so'],
    kwargs={'target': 26.8},
    input_core_dims=[['depth'], ['depth']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float]
).compute()

# Get depth of 26.8 isopycnal
depth_iso = xr.apply_ufunc(
    interp_to_isopycnal,
    day['sigma_theta'],
    day['depth'],
    kwargs={'target': 26.8},
    input_core_dims=[['depth'], ['depth']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float]
).compute()


In [ ]:
# Plot salinity over the 26.8 isopycnal for one day (jan 1 2020)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Salinity
c1 = ax1.pcolormesh(sal_iso['longitude'], sal_iso['latitude'], sal_iso, cmap='RdYlBu_r')
plt.colorbar(c1, ax=ax1, label='Salinity (psu)')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title('Salinity on σ₀=26.8 isopycnal, Jan 1 2020')

# Depth
c2 = ax2.pcolormesh(depth_iso['longitude'], depth_iso['latitude'], depth_iso, cmap='viridis_r')
cb = plt.colorbar(c2, ax=ax2, label='Depth (m)')
cb.ax.invert_yaxis()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Depth of σ₀=26.8 isopycnal, Jan 1 2020')

plt.tight_layout()
plt.show()